In [1]:
import torch
import torch.nn as nn

In [2]:
# Implement single head attention

class SingleHeadAttention(nn.Module):
    def __init__(self, input_embed_dim, embedding_dim, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_in = input_embed_dim
        self.d_out = embedding_dim
        self.context_length = context_length
        self.W_q = nn.Linear(self.d_in, self.d_out)
        self.W_k = nn.Linear(self.d_in, self.d_out)
        self.W_v = nn.Linear(self.d_in, self.d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(self.context_length, self.context_length), diagonal=1))
    # Query shape [batch_size, query_seq_length, embedding_dim]
    # Target shape [batch_size, query_seq_length, embedding_dim]
    # Output shape same as x
    # def forward(self, query, target): (What does it mean by target here?)
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_q(x)   # [b, num_tokens, d_out]
        keys = self.W_k(x)      # [b, num_tokens, d_out]
        values = self.W_v(x)    # [b, num_tokens, d_out]
        
        attn_scores = queries @ keys.transpose(2, 1)    # [b, num_tokens, num_tokens]
        attn_weights = torch.softmax((attn_scores.masked_fill(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf))/self.d_out**0.5, dim=-1) # [b, num_tokens, num_tokens]
        attn_weights = self.dropout(attn_weights)   # [b, num_tokens, num_tokens]
        
        context_vector = attn_weights @ values      # [n, num_tokens, d_out]
        
        return context_vector

In [2]:
# Implement multi head attention

class MultiheadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        """
        
        :param d_in: Input embed dim
        :param d_out: Output embed dimSame as embedding dim
        :param context_length: Total context window (maximum input num_tokens)
        :param dropout: Dropout ratio
        :param num_heads: Number of attention heads
        :param qkv_bias: Bias for q, k, v weight matrices (by default False)
        """
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.d_out = d_out
        self.W_q = nn.Linear(d_in, d_out, qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        
    def forward(self, x):
        """
        Batch size = b, context_length = cl, num_tokens = n, embed_length = el, num_heads = nh, head_dim = hd
        :param x: [b, n, d_in]
        :return: [b, n, d_out]
        """
        b, num_tokens, d_in = x.shape
        queries_pre = self.W_q(x)       # [bs, cl, d_out*nh]
        keys_pre = self.W_k(x)          # [bs, cl, d_out*nh]
        values_pre = self.W_v(x)        # [bs, cl, d_out*nh]
        
        # queries = [queries_pre[:,:,d_out*i:d_out:i+1] for i in range(self.num_heads)]
        queries = queries_pre.view(b, num_tokens, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # [b, nh, n, hd]
        # print(f"Q, K, V shape: {queries.shape}")    # [b, nh, n, hd]
        keys = keys_pre.reshape(b, num_tokens, self.num_heads, self.head_dim).permute(0, 2, 1, 3)   # [b, nh, n, hd]
        values = values_pre.reshape(b, num_tokens, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # [b, nh, n, hd]
        
        
        # keys_T = torch.transpose(torch.reshape(keys, (-1, self.context_length, self.d_out)), -2, -1)
        # print(f"Keys T shape: {keys_T.shape}")
        attn_scores = queries @ keys.transpose(2,3)         # [b, nh, n, n]
        # print(f"Attn scores shape: {attn_scores.shape}")    # [b, nh, n, n]
        
        # Context length and num_tokens may not be necessarily same
        # To save computation?
        attn_weights = torch.softmax(attn_scores.masked_fill(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)/self.d_out**0.5, dim = -1)
        attn_weights = self.dropout(attn_weights)       # [b, nh, n, n]
        context_vectors = attn_weights @ values  # [b, nh, n, hd]
        # print(f"Context vectors shape: {context_vectors.shape}") # [b, nh, n, hd]
        
        out = context_vectors.contiguous().permute(0, 2, 1, 3).view(b, num_tokens, self.d_out)      # [b, n, d_out]
        # print(f"Out shape: {out.shape}")    # [b, n, d_out]
        
        return out

In [3]:
# Implement row-wise feed forward layer (professor vala decompression/compression)

In [4]:
# Implement encoder block 

In [5]:
# Implement sequential encoder blocks

In [6]:
# Implement masked multi head attention